# LLaMA + RAG with Firecrawl Scrape (Web URL demo)

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** using:

- a local open-source LLM served via **Ollama** (for example, **LLaMA 3**),
- a lightweight vector database (**Chroma**) for retrieval, and
- **Firecrawl scrape** to turn a known URL into clean markdown before chunking and retrieval.

The workflow is intentionally simple and inspectable:

**known URL → Firecrawl scrape → chunks → embeddings → retrieval → grounded LLM answer**

> Tip for GitHub: keep the local Chroma folder ignored if you later add persistence.


## 0) Install dependencies

If you're in **Colab**, you may also want a terminal (`colab-xterm`) to run `ollama serve`.
If you're local, install Ollama normally and run the server in your terminal.


In [ ]:
# Core
!pip -q install -U ollama

# RAG stack (LangChain + Chroma)
!pip -q install -U langchain-community langchain-text-splitters langchain-chroma chromadb beautifulsoup4

# UI (optional)
!pip -q install -U gradio

# Firecrawl + env loading
!pip -q install -U firecrawl-py python-dotenv


### (Optional / Colab) Start Ollama server

If you're using Colab and want an in-notebook terminal:

1) run the next cell to enable `%xterm`  
2) in the xterm, start Ollama, for example:
   - `ollama serve`
   - then (once) `ollama pull llama3` and `ollama pull nomic-embed-text`


In [ ]:
# Optional: only needed on Colab
# !pip -q install colab-xterm
# %load_ext colabxterm
# %xterm


## 1) Configure models + Firecrawl

- `LLM_MODEL`: the chat model used to generate answers
- `EMBED_MODEL`: the embedding model used for vector search
- `URL`: the page we want to ground the assistant on

Set `FIRECRAWL_API_KEY` in `../../.env` or in your shell environment before running the notebook.


In [ ]:
LLM_MODEL = "llama3"
EMBED_MODEL = "nomic-embed-text"

# Seed URL to index
URL = "https://en.wikipedia.org/wiki/Ohiya"

# Optional: set FIRECRAWL_API_KEY in ../../.env or your shell environment


## 2) Quick smoke test: can we call the LLM?


In [ ]:
import ollama

resp = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence and tell me you are ready for RAG."}]
)
print(resp["message"]["content"])


## 3) Build the initial vector store from the seed URL with Firecrawl scrape


In [ ]:
import os
from dotenv import load_dotenv
from firecrawl import Firecrawl
from langchain_core.documents import Document

# Splitter import (newer LangChain puts splitters in langchain_text_splitters)
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# Chroma import (newer LangChain uses langchain_chroma)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

# Ollama embeddings import (community vs dedicated package depending on versions)
try:
    from langchain_ollama import OllamaEmbeddings
except Exception:
    from langchain_community.embeddings import OllamaEmbeddings

load_dotenv("../../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")
if not api_key:
    raise ValueError("Set FIRECRAWL_API_KEY in ../../.env or your shell environment before running this notebook.")

firecrawl = Firecrawl(api_key=api_key)

def _get_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

scrape_result = firecrawl.scrape(URL, formats=["markdown"])
markdown = _get_field(scrape_result, "markdown")
metadata = _get_field(scrape_result, "metadata", {}) or {}

if not markdown:
    raise ValueError(f"Firecrawl returned no markdown for {URL}")

docs = [
    Document(
        page_content=markdown,
        metadata={
            "source": URL,
            "title": metadata.get("title") if isinstance(metadata, dict) else None,
            "provider": "firecrawl",
        },
    )
]

chunks = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_firecrawl_web_demo",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Loaded seed URL via Firecrawl scrape: {URL}")
print(f"Seed docs: {len(docs)} | Seed chunks added: {len(chunks)}")


## 4) Define the grounded RAG answer function


In [ ]:
def rag_answer(question: str) -> str:
    try:
        retrieved = retriever.get_relevant_documents(question)
    except Exception:
        retrieved = retriever.invoke(question)

    context = "\n\n".join(d.page_content for d in retrieved)

    prompt = f"""You are a helpful assistant.
Use ONLY the context below to answer the question.
If the answer is not contained in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp["message"]["content"]


## 5) Test RAG in the notebook


In [ ]:
print(rag_answer("What is Ohiya? Give a short answer."))


## 6) Optional: Gradio UI

The UI shows the grounded answer for questions about the scraped URL.


In [ ]:
import gradio as gr

iface = gr.Interface(
    fn=rag_answer,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the loaded URL..."),
    outputs="text",
    title="RAG with Ollama (LLaMA) + Chroma + Firecrawl",
    description=f"URL loaded via Firecrawl scrape: {URL}"
)

iface.launch(debug=False)
